# Notebook 04 — Evaluación y Análisis de Resultados

Evaluación detallada del modelo entrenado: métricas, matriz de confusión, análisis de errores y ejemplos de predicción.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.train import train
from src.evaluate import evaluate, compute_metrics
from src.predict import FraudPredictor
from src.risk_analysis import analyze_risk
from src.config import PROCESSED_DATA_DIR, DEFAULT_MODEL

## 1. Cargar dataset y entrenar (o cargar modelo guardado)

In [ ]:
dataset_path = PROCESSED_DATA_DIR / 'messages.csv'

if dataset_path.exists():
    df = pd.read_csv(dataset_path)
else:
    print('Dataset no encontrado. Usando datos de demostración.')
    df = pd.DataFrame({
        'message': [
            'Your account is blocked verify at http://fake.com now',
            'Hello how are you doing today',
            'URGENT send your password and PIN immediately http://evil.com',
            'Reminder your appointment is tomorrow at 10am',
            'You won $10000 send card number to claim prize',
            'Thank you for your order it will arrive Friday',
            'Verify your identity now account suspended',
            'Your package could not be delivered reschedule now http://fake-delivery.com',
        ] * 50,
        'label': ['fraudulent', 'legitimate', 'fraudulent', 'legitimate',
                  'fraudulent', 'legitimate', 'fraudulent', 'fraudulent'] * 50
    })

result = train(df, model_name=DEFAULT_MODEL, save=True)
print(f'Modelo entrenado. Test set: {len(result["y_test"])} muestras.')

## 2. Evaluación completa

In [ ]:
metrics = evaluate(
    result['model'],
    result['X_test'],
    result['y_test'],
    result['int_to_label'],
    model_name=DEFAULT_MODEL,
)

print(f"\nAccuracy      : {metrics['accuracy']:.4f}")
print(f"Precision     : {metrics['precision']:.4f}")
print(f"Recall        : {metrics['recall']:.4f}")
print(f"F1-score      : {metrics['f1_score']:.4f}")
if 'recall_fraudulent' in metrics:
    print(f"Recall fraude : {metrics['recall_fraudulent']:.4f}  ← métrica crítica")

## 3. Análisis de errores

Mensajes clasificados incorrectamente (falsos negativos = fraudes no detectados).

In [ ]:
y_pred = result['model'].predict(result['X_test'])
y_test = result['y_test']
int_to_label = result['int_to_label']
test_texts = result['X_test_text']

# Falsos negativos: fraudes no detectados (los más peligrosos)
fraud_idx = next((k for k, v in int_to_label.items() if v == 'fraudulent'), None)

if fraud_idx is not None:
    errors = [
        {'text': t, 'true': int_to_label[int(yt)], 'pred': int_to_label[int(yp)]}
        for t, yt, yp in zip(test_texts, y_test, y_pred)
        if yt != yp
    ]
    fn = [e for e in errors if e['true'] == 'fraudulent']
    fp = [e for e in errors if e['pred'] == 'fraudulent']

    print(f'Total de errores: {len(errors)}')
    print(f'Falsos negativos (fraude no detectado): {len(fn)}')
    print(f'Falsos positivos (legítimo clasificado como fraude): {len(fp)}')

    if fn:
        print('\nEjemplos de fraudes no detectados (primeros 3):')
        for e in fn[:3]:
            print(f'  "{e["text"][:100]}"')

## 4. Demo de predicción con ejemplos en español

In [ ]:
from app.examples import EXAMPLES
from src.utils import setup_logging
setup_logging()

try:
    predictor = FraudPredictor()
    print('Modelo cargado exitosamente.\n')

    for ex in EXAMPLES[:5]:
        r = predictor.predict(ex['message'])
        match = '✓' if r['predicted_class'] == ex['expected_category'] else '✗'
        print(f"{match} [{ex['title']}]")
        print(f"   Predicción : {r['predicted_class'].upper()} (esperado: {ex['expected_category'].upper()})")
        print(f"   Riesgo     : {r['risk_level'].upper()} | Score: {r['risk_score']}/100")
        if r['signals']:
            print(f"   Señales    : {', '.join(r['signals'][:3])}")
        print()
except Exception as e:
    print(f'No se pudo cargar el predictor: {e}')
    print('Ejecuta primero: python main.py train --dataset data/processed/messages.csv')

## 5. Análisis de riesgo basado en reglas (sin modelo)

In [ ]:
test_messages = [
    'ALERTA: Su cuenta será BLOQUEADA. Verifique en http://banco-falso.com ahora.',
    'Hola, ¿cómo estás? Te llamo mañana para coordinar.',
    'Transferimos $5,000 a su cuenta. Confirme su PIN y contraseña.',
    'Felicidades, ganó el sorteo. Envíe su número de tarjeta.',
]

for msg in test_messages:
    r = analyze_risk(msg)
    print(f'Mensaje  : {msg[:70]}...' if len(msg) > 70 else f'Mensaje  : {msg}')
    print(f'Score    : {r["risk_score"]}/100 | Nivel: {r["risk_level"].upper()}')
    print(f'Señales  : {r["signals"]}')
    print()